In [1]:
import sys, os
# ensure parent directory is on the path so `src` package can be imported
sys.path.insert(0, os.path.abspath('..'))

In [2]:
# configura per importare da src
import sys
sys.path.append('./src')

In [ ]:
from src.CBEM.model import BoxEmbeddingCBM
from src.CBEM.train import train_and_validate, plot_train_val_history
from src.utils.dataset import classical_split_awa2_features
from torch import optim
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

AllenNLP not available. Registrable won't work.


In [4]:
# leggi matrice V_gt da file csv
V_gt = pd.read_csv('../AwA2_Dataset_Labels/Animals_with_Attributes2/V_gt.csv', sep='\t', index_col=0).values
V_gt = torch.tensor(V_gt, dtype=torch.float32)

In [7]:
V_gt.shape

torch.Size([50, 50])

In [5]:
features_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-features.txt'
labels_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-labels.txt'

(X_train, y_train), (X_val, y_val), (X_test, y_test) = classical_split_awa2_features(
    features_path, labels_path, test_size=0.2, val_size=0.1, random_seed=42
)

Caricamento dei dati in corso... (potrebbe richiedere qualche secondo)
Dataset caricato correttamente: 37322 campioni con 2048 feature ciascuno.

--- Risultati dello Split Stratificato (50 Classi) ---
Training set:   26124 campioni
Validation set: 3733 campioni
Test set:       7465 campioni


In [6]:
class_concept_matrix = torch.from_numpy(np.loadtxt('../Awa2_Dataset_Labels/Animals_with_Attributes2/extended_matrix.txt', dtype=int))

In [8]:
class_concept_matrix.shape

torch.Size([50, 50])

In [ ]:
LATENT_DIM = X_train.shape[1]
print(f"Dimensione features: {LATENT_DIM}")
NUM_CONCEPTS = V_gt.shape[0]  # numero di classi è dato dal numero di righe di V_gt
print(f"Numero di concetti: {NUM_CONCEPTS}")
NUM_CLASSES = len(set(y_train))  # numero di classi è dato dal numero di classi uniche nei label di training
print(f"Numero di classi: {NUM_CLASSES}")

# Iperparametri del modello
BOX_DIM = 16
BATCH_SIZE = 32
EPOCHS = 20
    
model = BoxEmbeddingCBM(LATENT_DIM, NUM_CONCEPTS, BOX_DIM, NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = TensorDataset(X_val, y_val)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    
train_val_history = train_and_validate(model, train_dataloader, val_dataloader, optimizer, class_concept_matrix, V_gt, EPOCHS, device)

In [ ]:
plot_train_val_history(train_val_history)